# YOLO Object Detection Baseline
## Processed 데이터 → 학습 → 검증 → 테스트 → 성능 시각화

이 노트북은 전처리 담당자가 전달한 **YOLO Detection 형식의 `processed` 데이터**를 받아,
추가 전처리나 증강 실험 없이 baseline 모델의 전체 파이프라인을 수행하기 위한 파일입니다.

### 전체 흐름

1. 실행 환경과 GPU 확인
2. `processed/data.yaml` 연결
3. Train / Validation / Test 구조 확인
4. YOLO 라벨 sanity check
5. 클래스 분포와 split 간 데이터 누수 확인
6. pretrained YOLO nano baseline 학습
7. 학습 loss와 metric curve 확인
8. `best.pt`로 Validation 정식 평가
9. Precision / Recall / mAP50 / mAP50-95 시각화
10. Confusion Matrix / PR Curve / 클래스별 AP 확인
11. Test set 최종 평가
12. Validation / Test 성능 비교
13. Test 이미지 prediction 직접 확인
14. 최종 결과 CSV 저장

### 이 노트북의 Baseline 정의

- pretrained `yolo26n.pt`
- 입력 크기 `640`
- 별도의 커스텀 augmentation 조합을 추가하지 않음
- Ultralytics 기본 학습 동작 유지
- 동일한 Train / Validation / Test split 사용
- 평가에는 `best.pt` 사용

> Validation은 학습 상태 확인과 모델 선택에 사용합니다.  
> Test set은 baseline 설정이 확정된 뒤 **최종 일반화 성능을 확인하는 용도**로 사용합니다.

# 1. 전처리 담당자에게 받아야 하는 데이터 구조

권장 구조:

```text
processed/
├── data.yaml
├── images/
│   ├── train/
│   ├── val/
│   └── test/
└── labels/
    ├── train/
    ├── val/
    └── test/
```

이미지와 라벨은 같은 stem을 사용합니다.

```text
images/train/sample.jpg
labels/train/sample.txt
```

YOLO Detection 라벨 한 줄:

```text
class_id x_center y_center width height
```

`x_center`, `y_center`, `width`, `height`는 모두 `0 ~ 1` 범위로 정규화되어 있어야 합니다.

`data.yaml` 예:

```yaml
path: .
train: images/train
val: images/val
test: images/test

names:
  0: class_0
  1: class_1
```

이 노트북은 학습/검증/테스트를 모두 수행하므로 `test:` split이 반드시 필요합니다.

# 2. 라이브러리 불러오기

이 노트북에는 설치 명령을 넣지 않습니다.
현재 프로젝트 Python 환경에 준비된 라이브러리를 그대로 사용합니다.

In [ ]:
from __future__ import annotations

import sys
import time
import random
import hashlib
import platform
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import yaml
import torch

from ultralytics import YOLO
from IPython.display import display, Image as IPImage

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

def set_korean_font():
    candidates = ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR"]
    installed = {f.name for f in fm.fontManager.ttflist}
    for name in candidates:
        if name in installed:
            plt.rcParams["font.family"] = name
            break
    plt.rcParams["axes.unicode_minus"] = False

set_korean_font()

print("Python :", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("OpenCV :", cv2.__version__)
print("OS     :", platform.platform())

# 3. Seed 고정

Baseline끼리 비교할 때 불필요한 난수 차이를 줄이기 위해 seed를 고정합니다.

딥러닝 특성상 하드웨어나 CUDA 연산에 따라 결과가 완전히 동일하지 않을 수 있지만,
실험 재현성을 높이는 데 도움이 됩니다.

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("SEED =", SEED)

# 4. GPU / Device 확인

우선순위는 CUDA GPU → Apple MPS → CPU입니다.

학습 시작 전에 반드시 출력 결과에서 원하는 GPU가 잡혔는지 확인하세요.

In [ ]:
if torch.cuda.is_available():
    DEVICE = 0
    print("CUDA GPU:", torch.cuda.get_device_name(0))
    memory_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"VRAM: {memory_gb:.2f} GB")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
    print("Apple MPS 사용")
else:
    DEVICE = "cpu"
    print("CPU 사용")

print("DEVICE =", DEVICE)

# 5. Baseline 설정

가장 먼저 `PROCESSED_DIR`만 실제 경로로 수정하세요.

### 주요 파라미터

- `MODEL_NAME`: baseline pretrained 모델
- `EPOCHS`: 최대 학습 epoch
- `IMGSZ`: 모델 입력 크기
- `BATCH`: batch size
- `PATIENCE`: Validation 성능 개선이 없을 때 기다리는 epoch 수
- `WORKERS=0`: Windows + Jupyter에서 multiprocessing 문제를 줄이기 위한 안전한 설정

Baseline에서는 `mosaic`, `hsv`, `degrees` 같은 augmentation 파라미터를 직접 덮어쓰지 않습니다.

In [ ]:
# Windows 예:
# PROCESSED_DIR = Path(r"C:\ai_challingers\recycling_ssg\data\processed")

PROCESSED_DIR = Path("data/processed").resolve()
DATA_YAML = PROCESSED_DIR / "data.yaml"

MODEL_NAME = "yolo26n.pt"
EPOCHS = 100
IMGSZ = 640
BATCH = 16
PATIENCE = 20
WORKERS = 0

EXPERIMENT_ROOT = Path("yolo_baseline_results").resolve()
TRAIN_PROJECT = EXPERIMENT_ROOT / "train"
EVAL_PROJECT = EXPERIMENT_ROOT / "evaluation"
PREDICT_PROJECT = EXPERIMENT_ROOT / "predictions"
RUN_NAME = "yolo26n_baseline"

for p in [EXPERIMENT_ROOT, TRAIN_PROJECT, EVAL_PROJECT, PREDICT_PROJECT]:
    p.mkdir(parents=True, exist_ok=True)

print("PROCESSED_DIR:", PROCESSED_DIR)
print("DATA_YAML    :", DATA_YAML)
print("MODEL_NAME   :", MODEL_NAME)
print("EPOCHS       :", EPOCHS)
print("IMGSZ        :", IMGSZ)
print("BATCH        :", BATCH)
print("RESULT ROOT  :", EXPERIMENT_ROOT)

# 6. `data.yaml` 읽기

학습 전에 다음 항목이 모두 있는지 검사합니다.

- `train`
- `val`
- `test`
- `names`

Test split이 없다면 최종 성능 평가가 불가능하므로 여기서 중단합니다.

In [ ]:
if not DATA_YAML.exists():
    raise FileNotFoundError(
        f"data.yaml을 찾을 수 없습니다: {DATA_YAML}\n"
        "PROCESSED_DIR를 실제 processed 폴더로 수정하세요."
    )

with open(DATA_YAML, "r", encoding="utf-8") as f:
    data_config = yaml.safe_load(f)

required_keys = ["train", "val", "test", "names"]
missing = [k for k in required_keys if k not in data_config]

if missing:
    raise KeyError(f"data.yaml에 필요한 항목이 없습니다: {missing}")

names_raw = data_config["names"]

if isinstance(names_raw, dict):
    CLASS_NAMES = {int(k): str(v) for k, v in names_raw.items()}
elif isinstance(names_raw, list):
    CLASS_NAMES = {i: str(v) for i, v in enumerate(names_raw)}
else:
    raise TypeError("data.yaml의 names는 dict 또는 list 형식이어야 합니다.")

NUM_CLASSES = len(CLASS_NAMES)

print(yaml.safe_dump(data_config, allow_unicode=True, sort_keys=False))
print("Number of classes:", NUM_CLASSES)

display(pd.DataFrame({
    "class_id": list(CLASS_NAMES.keys()),
    "class_name": list(CLASS_NAMES.values()),
}))

# 7. Train / Validation / Test 이미지 목록 만들기

`data.yaml`의 `path`와 각 split 경로를 실제 파일 경로로 해석합니다.

이미지 디렉터리와 이미지 경로가 적힌 `.txt` 목록 파일을 모두 지원합니다.

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

def get_dataset_root(data_yaml: Path, config: dict) -> Path:
    base = data_yaml.parent.resolve()
    root_value = config.get("path")
    if root_value is None:
        return base
    root = Path(root_value)
    return root.resolve() if root.is_absolute() else (base / root).resolve()

DATASET_ROOT = get_dataset_root(DATA_YAML, data_config)

def resolve_split_entry(entry, root: Path):
    entries = entry if isinstance(entry, list) else [entry]
    files = []

    for item in entries:
        p = Path(item)
        if not p.is_absolute():
            p = (root / p).resolve()

        if p.is_dir():
            files.extend(
                x for x in p.rglob("*")
                if x.is_file() and x.suffix.lower() in IMAGE_EXTENSIONS
            )
        elif p.is_file() and p.suffix.lower() == ".txt":
            for line in p.read_text(encoding="utf-8").splitlines():
                line = line.strip()
                if not line:
                    continue
                x = Path(line)
                if not x.is_absolute():
                    x = (root / x).resolve()
                files.append(x)
        elif p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS:
            files.append(p)
        else:
            raise FileNotFoundError(f"Split 경로를 찾을 수 없습니다: {p}")

    return sorted(set(x.resolve() for x in files))

split_files = {
    split: resolve_split_entry(data_config[split], DATASET_ROOT)
    for split in ["train", "val", "test"]
}

print("DATASET_ROOT:", DATASET_ROOT)
for split, files in split_files.items():
    print(f"{split:>5}: {len(files):,} images")

# 8. 이미지와 라벨 파일 연결

일반적인 YOLO 구조처럼 경로의 `images` 부분을 `labels`로 바꾸고,
확장자를 `.txt`로 변경하여 label 파일을 찾습니다.

In [ ]:
def image_to_label_path(image_path: Path) -> Path:
    parts = list(image_path.parts)
    indices = [i for i, part in enumerate(parts) if part.lower() == "images"]

    if not indices:
        raise ValueError(f"'images' 폴더를 포함하지 않는 경로입니다: {image_path}")

    parts[indices[-1]] = "labels"
    return Path(*parts).with_suffix(".txt")

examples = []
for split in ["train", "val", "test"]:
    for img in split_files[split][:2]:
        examples.append({
            "split": split,
            "image": str(img),
            "label": str(image_to_label_path(img)),
        })

display(pd.DataFrame(examples))

# 9. YOLO 라벨 Sanity Check

전처리 데이터를 수정하지 않고 학습 가능한 상태인지 검사합니다.

검사 항목:

- label 파일 존재 여부
- 라벨 한 줄이 정확히 5개 값인지
- class id가 정수인지
- class id가 `data.yaml`의 `names`에 존재하는지
- bbox 값이 0~1 범위인지
- width / height가 0보다 큰지
- split별 이미지 수 / 객체 수

오류가 발견되면 모델 학습 전에 중단합니다.

In [ ]:
def audit_split(split, image_files):
    rows = []
    errors = []
    counter = Counter()

    for image_path in image_files:
        label_path = image_to_label_path(image_path)
        row = {
            "split": split,
            "image_path": str(image_path),
            "label_exists": label_path.exists(),
            "objects": 0,
            "empty_label": False,
        }

        if not label_path.exists():
            errors.append({
                "split": split,
                "image": str(image_path),
                "error": "label file missing",
            })
            rows.append(row)
            continue

        text = label_path.read_text(encoding="utf-8").strip()

        if not text:
            row["empty_label"] = True
            rows.append(row)
            continue

        for line_no, line in enumerate(text.splitlines(), 1):
            try:
                values = line.split()
                if len(values) != 5:
                    raise ValueError(f"expected 5 values, got {len(values)}")

                cls_float, xc, yc, bw, bh = map(float, values)
                cls_id = int(cls_float)

                if cls_float != cls_id:
                    raise ValueError("class id is not integer")
                if cls_id not in CLASS_NAMES:
                    raise ValueError(f"class id {cls_id} not in names")

                coords = np.array([xc, yc, bw, bh], dtype=float)
                if not np.all((coords >= 0) & (coords <= 1)):
                    raise ValueError(f"bbox out of 0~1 range: {coords.tolist()}")
                if bw <= 0 or bh <= 0:
                    raise ValueError("bbox width/height must be > 0")

                row["objects"] += 1
                counter[cls_id] += 1

            except Exception as e:
                errors.append({
                    "split": split,
                    "image": str(image_path),
                    "label": str(label_path),
                    "line": line_no,
                    "content": line,
                    "error": repr(e),
                })

        rows.append(row)

    return pd.DataFrame(rows), pd.DataFrame(errors), counter

audit_parts = []
error_parts = []
class_counts_by_split = {}

for split in ["train", "val", "test"]:
    rows, errors, counter = audit_split(split, split_files[split])
    audit_parts.append(rows)
    if len(errors):
        error_parts.append(errors)
    class_counts_by_split[split] = counter

audit_df = pd.concat(audit_parts, ignore_index=True)
audit_error_df = pd.concat(error_parts, ignore_index=True) if error_parts else pd.DataFrame()

audit_summary = (
    audit_df.groupby("split")
    .agg(
        images=("image_path", "count"),
        labels=("label_exists", "sum"),
        empty_labels=("empty_label", "sum"),
        objects=("objects", "sum"),
    )
    .reindex(["train", "val", "test"])
)

display(audit_summary)

if len(audit_error_df):
    display(audit_error_df.head(30))
    raise ValueError("YOLO 라벨 sanity check에서 오류가 발견되었습니다.")

print("Label sanity check: PASSED")

# 10. 클래스별 객체 수 확인

전체 mAP를 해석할 때 각 클래스의 데이터 수를 함께 봐야 합니다.

특히 Validation이나 Test에 객체가 매우 적은 클래스는
per-class AP가 표본 수 때문에 크게 흔들릴 수 있습니다.

In [ ]:
support_rows = []

for class_id, class_name in CLASS_NAMES.items():
    support_rows.append({
        "class_id": class_id,
        "class_name": class_name,
        "train_objects": class_counts_by_split["train"].get(class_id, 0),
        "val_objects": class_counts_by_split["val"].get(class_id, 0),
        "test_objects": class_counts_by_split["test"].get(class_id, 0),
    })

class_support_df = pd.DataFrame(support_rows)
class_support_df["total_objects"] = class_support_df[
    ["train_objects", "val_objects", "test_objects"]
].sum(axis=1)

display(class_support_df)

print("Train에 없는 클래스:", int((class_support_df["train_objects"] == 0).sum()))
print("Validation에 없는 클래스:", int((class_support_df["val_objects"] == 0).sum()))
print("Test에 없는 클래스:", int((class_support_df["test_objects"] == 0).sum()))

In [ ]:
plot_df = class_support_df.sort_values("train_objects")

plt.figure(figsize=(12, max(8, len(plot_df) * 0.25)))
y = np.arange(len(plot_df))

plt.barh(y, plot_df["train_objects"], label="Train")
plt.barh(y, plot_df["val_objects"], left=plot_df["train_objects"], label="Validation")
plt.barh(
    y,
    plot_df["test_objects"],
    left=plot_df["train_objects"] + plot_df["val_objects"],
    label="Test",
)

plt.yticks(y, plot_df["class_name"])
plt.xlabel("Object count")
plt.title("Class Support by Split")
plt.legend()
plt.tight_layout()
plt.show()

# 11. Train / Validation / Test 데이터 누수 확인

같은 이미지가 서로 다른 split에 들어가면 평가 결과가 실제보다 높아질 수 있습니다.

파일명 대신 SHA-1 hash를 계산하여 **바이트가 완전히 동일한 이미지**가
Train / Validation / Test에 동시에 들어가 있는지 확인합니다.

데이터가 매우 크다면 이 셀은 다소 시간이 걸릴 수 있습니다.

In [ ]:
CHECK_HASH_DUPLICATES = True

def sha1_file(path: Path, chunk_size=1024 * 1024):
    h = hashlib.sha1()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

if CHECK_HASH_DUPLICATES:
    rows = []

    for split, files in split_files.items():
        for path in files:
            rows.append({
                "split": split,
                "image_path": str(path),
                "sha1": sha1_file(path),
            })

    hash_df = pd.DataFrame(rows)

    leakage = (
        hash_df.groupby("sha1")
        .filter(lambda g: g["split"].nunique() > 1)
        .sort_values(["sha1", "split"])
    )

    print("Cross-split exact duplicate rows:", len(leakage))

    if len(leakage):
        display(leakage)
        raise ValueError("Train / Validation / Test 사이에 동일 이미지가 발견되었습니다.")

    print("Data leakage check: PASSED")

# 12. Ground Truth bbox를 직접 확인

숫자 검증만 통과했다고 해서 bbox가 실제 객체를 잘 감싸고 있다는 뜻은 아닙니다.

각 split에서 샘플 이미지를 직접 그려 다음을 확인합니다.

- bbox 위치가 맞는가?
- bbox가 지나치게 크거나 작지 않은가?
- class id가 올바른가?
- 이미지와 label이 서로 맞는가?

OpenCV 기본 폰트는 한글 표시가 제한되므로 이미지에는 class id만 표시하고,
아래 표에서 클래스명을 확인합니다.

In [ ]:
def load_yolo_boxes(image_path: Path):
    image = cv2.imread(str(image_path))
    if image is None:
        raise ValueError(f"이미지를 읽을 수 없습니다: {image_path}")

    h, w = image.shape[:2]
    boxes, classes = [], []

    label_path = image_to_label_path(image_path)
    text = label_path.read_text(encoding="utf-8").strip()

    for line in text.splitlines():
        cls_id, xc, yc, bw, bh = map(float, line.split())
        cls_id = int(cls_id)

        x1 = (xc - bw / 2) * w
        y1 = (yc - bh / 2) * h
        x2 = (xc + bw / 2) * w
        y2 = (yc + bh / 2) * h

        boxes.append([x1, y1, x2, y2])
        classes.append(cls_id)

    return image, np.asarray(boxes, dtype=float).reshape(-1, 4), np.asarray(classes, dtype=int)

def draw_gt(image, boxes, classes):
    out = image.copy()

    for box, cls_id in zip(boxes, classes):
        x1, y1, x2, y2 = map(int, box)

        cv2.rectangle(out, (x1, y1), (x2, y2), (0, 255, 0), 3)

        cv2.putText(
            out,
            str(cls_id),
            (x1, max(20, y1 - 8)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2,
            cv2.LINE_AA,
        )

    return out

NUM_GT_SAMPLES = 2

for split in ["train", "val", "test"]:
    print("=" * 70, split.upper())

    for image_path in split_files[split][:NUM_GT_SAMPLES]:
        image, boxes, classes = load_yolo_boxes(image_path)

        plt.figure(figsize=(10, 7))
        plt.imshow(cv2.cvtColor(draw_gt(image, boxes, classes), cv2.COLOR_BGR2RGB))
        plt.title(f"{split}: {image_path.name}")
        plt.axis("off")
        plt.tight_layout()
        plt.show()

        display(pd.DataFrame({
            "class_id": classes,
            "class_name": [CLASS_NAMES[int(c)] for c in classes],
        }))

# 13. YOLO Baseline 모델 로드

pretrained `yolo26n.pt`에서 시작합니다.

Nano 모델은 작고 빠르기 때문에 첫 기준 모델로 사용하기 좋으며,
이후 더 큰 모델이나 증강 조합의 효과를 비교하기 쉽습니다.

In [ ]:
model = YOLO(MODEL_NAME)
print(model)

# 14. Baseline 학습

이 셀이 실제 학습을 수행합니다.

### 전달하는 주요 인자

- `data`: `data.yaml`
- `epochs`: 최대 epoch
- `imgsz`: 입력 크기
- `batch`: batch size
- `device`: GPU / MPS / CPU
- `patience`: Early Stopping
- `seed`: 재현성
- `deterministic=True`: 가능한 범위에서 결정적 연산 사용
- `optimizer="auto"`: Ultralytics가 학습 조건에 맞는 optimizer 선택
- `plots=True`: 학습 curve와 시각화 파일 저장

Baseline 비교를 위해 augmentation 관련 인자는 임의로 변경하지 않습니다.

In [ ]:
train_started = time.perf_counter()

train_results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    device=DEVICE,
    workers=WORKERS,
    seed=SEED,
    deterministic=True,
    optimizer="auto",
    project=str(TRAIN_PROJECT),
    name=RUN_NAME,
    exist_ok=True,
    plots=True,
    verbose=True,
)

train_minutes = (time.perf_counter() - train_started) / 60.0

print(f"Training time: {train_minutes:.2f} minutes")

# 15. `best.pt`와 `last.pt` 확인

학습이 끝나면 일반적으로 다음 checkpoint가 저장됩니다.

- `best.pt`: 학습 중 Validation 기준 가장 좋았던 checkpoint
- `last.pt`: 마지막 epoch의 checkpoint

Validation과 Test 평가는 `best.pt`를 사용합니다.

In [ ]:
TRAIN_SAVE_DIR = Path(train_results.save_dir)
BEST_PT = TRAIN_SAVE_DIR / "weights" / "best.pt"
LAST_PT = TRAIN_SAVE_DIR / "weights" / "last.pt"

print("TRAIN_SAVE_DIR:", TRAIN_SAVE_DIR)
print("BEST_PT       :", BEST_PT)
print("LAST_PT       :", LAST_PT)

if not BEST_PT.exists():
    raise FileNotFoundError(f"best.pt를 찾을 수 없습니다: {BEST_PT}")

# 16. Ultralytics 전체 학습 결과 그림 확인

`plots=True`로 학습하면 일반적으로 `results.png`가 저장됩니다.

이 그림에서 train/val loss와 Precision, Recall, mAP 흐름을 빠르게 확인할 수 있습니다.

In [ ]:
TRAIN_RESULTS_PNG = TRAIN_SAVE_DIR / "results.png"

if TRAIN_RESULTS_PNG.exists():
    display(IPImage(filename=str(TRAIN_RESULTS_PNG), width=1200))
else:
    print("results.png를 찾을 수 없습니다:", TRAIN_RESULTS_PNG)

# 17. `results.csv` 읽기

YOLO는 epoch별 결과를 CSV로 저장합니다.

이 파일을 직접 읽으면 특정 epoch의 수치를 확인하거나
원하는 형태의 그래프를 직접 만들 수 있습니다.

In [ ]:
TRAIN_RESULTS_CSV = TRAIN_SAVE_DIR / "results.csv"

if not TRAIN_RESULTS_CSV.exists():
    raise FileNotFoundError(f"results.csv를 찾을 수 없습니다: {TRAIN_RESULTS_CSV}")

history_df = pd.read_csv(TRAIN_RESULTS_CSV)
history_df.columns = [c.strip() for c in history_df.columns]

display(history_df.tail())

print("Columns:")
for col in history_df.columns:
    print("-", col)

# 18. Train / Validation Loss 시각화

Detection 모델에서는 주로 다음 loss를 확인합니다.

- Box loss: bbox 위치 관련
- Classification loss: 클래스 분류 관련
- DFL loss: bbox regression 품질 관련

일반적으로 Train loss와 Validation loss가 함께 감소하는 흐름이 바람직합니다.
Train loss만 낮아지고 Validation loss가 다시 증가하면 과적합 가능성을 확인합니다.

In [ ]:
def find_history_column(df, *keywords):
    for col in df.columns:
        lowered = col.lower()
        if all(keyword.lower() in lowered for keyword in keywords):
            return col
    return None

loss_specs = [
    ("Box Loss", "box_loss"),
    ("Classification Loss", "cls_loss"),
    ("DFL Loss", "dfl_loss"),
]

for title, keyword in loss_specs:
    train_col = find_history_column(history_df, "train", keyword)
    val_col = find_history_column(history_df, "val", keyword)

    if train_col is None and val_col is None:
        print("Skip:", title)
        continue

    plt.figure(figsize=(10, 5))

    if train_col:
        plt.plot(history_df.index + 1, history_df[train_col], label="Train")
    if val_col:
        plt.plot(history_df.index + 1, history_df[val_col], label="Validation")

    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

# 19. Validation 성능 지표의 epoch별 변화

다음 지표를 각각 확인합니다.

- Precision
- Recall
- mAP50
- mAP50-95

Baseline 모델을 비교할 때는 **mAP50-95를 1차 기준**으로 보는 것을 권장합니다.

mAP50은 IoU 0.5 기준이라 상대적으로 관대하지만,
mAP50-95는 여러 IoU threshold를 평균하므로 bbox 위치 정확도까지 더 엄격하게 반영합니다.

In [ ]:
metric_specs = [
    ("Precision", ["precision"]),
    ("Recall", ["recall"]),
    ("mAP50", ["map50"]),
    ("mAP50-95", ["map50-95"]),
]

for title, keywords in metric_specs:
    col = find_history_column(history_df, *keywords)

    if col is None:
        print("Skip:", title)
        continue

    plt.figure(figsize=(10, 5))
    plt.plot(history_df.index + 1, history_df[col])
    plt.xlabel("Epoch")
    plt.ylabel(title)
    plt.title(f"Validation {title}")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

# 20. 가장 좋은 checkpoint 다시 로드

학습이 끝난 뒤 저장된 `best.pt`를 새 YOLO 객체로 다시 불러옵니다.

이렇게 하면 실제 배포나 팀원 전달 상황처럼
**저장된 weight 파일만으로 정상적으로 사용할 수 있는지**도 함께 확인할 수 있습니다.

In [ ]:
best_model = YOLO(str(BEST_PT))
print("Loaded:", BEST_PT)

# 21. Validation set 정식 평가

저장된 `best.pt`로 Validation set을 명시적으로 다시 평가합니다.

`plots=True`로 설정하면 다음 시각 자료가 저장됩니다.

- Confusion Matrix
- Normalized Confusion Matrix
- PR Curve
- F1 Curve
- Precision Curve
- Recall Curve
- Validation prediction sample

In [ ]:
val_metrics = best_model.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    plots=True,
    project=str(EVAL_PROJECT),
    name="validation",
    exist_ok=True,
    verbose=True,
)

VAL_SAVE_DIR = Path(val_metrics.save_dir)
print("Validation outputs:", VAL_SAVE_DIR)

# 22. Validation 핵심 성능 수치 정리

### Precision
모델이 객체라고 예측한 것 중 실제 정답인 비율입니다.
높을수록 False Positive가 적습니다.

### Recall
실제 객체 중 모델이 찾아낸 비율입니다.
높을수록 놓친 객체가 적습니다.

### mAP50
IoU 0.5 기준 AP의 클래스 평균입니다.

### mAP50-95
IoU 0.50~0.95 여러 기준의 AP를 평균합니다.
bbox 위치까지 더 엄격하게 평가하므로 대표 Detection 성능 지표로 사용하기 좋습니다.

In [ ]:
def metrics_to_row(metrics, split_name):
    box = metrics.box

    precision = float(getattr(box, "mp", np.nan))
    recall = float(getattr(box, "mr", np.nan))

    f1 = (
        2 * precision * recall / (precision + recall)
        if np.isfinite(precision)
        and np.isfinite(recall)
        and (precision + recall) > 0
        else np.nan
    )

    speed = getattr(metrics, "speed", {}) or {}

    return {
        "split": split_name,
        "precision": precision,
        "recall": recall,
        "f1_from_global_P_R": f1,
        "mAP50": float(box.map50),
        "mAP75": float(box.map75),
        "mAP50_95": float(box.map),
        "inference_ms_per_image": speed.get("inference", np.nan),
    }

val_summary = pd.DataFrame([metrics_to_row(val_metrics, "validation")])

display(val_summary.style.format({
    "precision": "{:.4f}",
    "recall": "{:.4f}",
    "f1_from_global_P_R": "{:.4f}",
    "mAP50": "{:.4f}",
    "mAP75": "{:.4f}",
    "mAP50_95": "{:.4f}",
    "inference_ms_per_image": "{:.3f}",
}))

# 23. Validation 성능 Bar Chart

표의 핵심 수치를 그래프로도 확인합니다.

팀 공유나 발표에서는 mAP50-95 수치와 함께 이 그래프를 사용하면 이해하기 쉽습니다.

In [ ]:
row = val_summary.iloc[0]

metric_names = ["Precision", "Recall", "mAP50", "mAP50-95"]
metric_values = [row["precision"], row["recall"], row["mAP50"], row["mAP50_95"]]

plt.figure(figsize=(9, 5))
bars = plt.bar(metric_names, metric_values)
plt.ylim(0, 1)
plt.ylabel("Score")
plt.title("YOLO Baseline - Validation Metrics")

for bar, value in zip(bars, metric_values):
    if np.isfinite(value):
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            value + 0.02,
            f"{value:.4f}",
            ha="center",
        )

plt.tight_layout()
plt.show()

# 24. Validation PR Curve / Confusion Matrix 확인

### PR Curve
Precision과 Recall의 trade-off를 보여줍니다.
곡선이 오른쪽 위에 가까울수록 좋습니다.

### Confusion Matrix
실제 클래스와 예측 클래스가 어떻게 혼동되는지 보여줍니다.
서로 비슷한 폐기물 클래스가 자주 혼동되는지 확인할 때 특히 유용합니다.

In [ ]:
def display_saved_image(folder: Path, filename: str, width=1100):
    path = folder / filename

    if path.exists():
        print(filename)
        display(IPImage(filename=str(path), width=width))
    else:
        print("Not found:", path)

validation_plot_files = [
    "PR_curve.png",
    "F1_curve.png",
    "P_curve.png",
    "R_curve.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "val_batch0_pred.jpg",
]

for filename in validation_plot_files:
    display_saved_image(VAL_SAVE_DIR, filename)

# 25. Validation Confusion Matrix를 표로 저장

현재 Ultralytics 버전에서 `ConfusionMatrix.to_df()`를 지원하면
Confusion Matrix를 CSV로도 저장합니다.

이미지뿐 아니라 표를 남겨두면 특정 클래스 오류 분석에 활용하기 좋습니다.

In [ ]:
VAL_CONFUSION_CSV = EXPERIMENT_ROOT / "validation_confusion_matrix.csv"

if hasattr(val_metrics, "confusion_matrix") and hasattr(val_metrics.confusion_matrix, "to_df"):
    val_confusion_df = val_metrics.confusion_matrix.to_df()
    display(val_confusion_df)

    if hasattr(val_confusion_df, "write_csv"):
        val_confusion_df.write_csv(VAL_CONFUSION_CSV)
    elif hasattr(val_confusion_df, "to_csv"):
        val_confusion_df.to_csv(VAL_CONFUSION_CSV, index=False, encoding="utf-8-sig")

    print("Saved:", VAL_CONFUSION_CSV)
else:
    print("현재 버전에서는 confusion_matrix.to_df()를 사용할 수 없습니다.")

# 26. Validation 클래스별 mAP50-95

전체 평균이 괜찮아도 일부 클래스가 거의 탐지되지 않을 수 있습니다.

각 클래스의 Validation mAP50-95와 해당 split의 객체 수를 함께 확인합니다.
support가 매우 작은 클래스의 AP는 과도하게 해석하지 않는 것이 좋습니다.

In [ ]:
val_maps = np.asarray(val_metrics.box.maps, dtype=float)

val_per_class_df = pd.DataFrame({
    "class_id": range(len(val_maps)),
    "class_name": [CLASS_NAMES.get(i, f"class_{i}") for i in range(len(val_maps))],
    "val_mAP50_95": val_maps,
})

val_per_class_df = val_per_class_df.merge(
    class_support_df[
        ["class_id", "train_objects", "val_objects", "test_objects"]
    ],
    on="class_id",
    how="left",
).sort_values("val_mAP50_95", ascending=False)

display(val_per_class_df)

In [ ]:
plot_df = val_per_class_df.sort_values("val_mAP50_95")

plt.figure(figsize=(12, max(8, len(plot_df) * 0.25)))
plt.barh(plot_df["class_name"], plot_df["val_mAP50_95"])
plt.xlim(0, 1)
plt.xlabel("Validation mAP50-95")
plt.ylabel("Class")
plt.title("Per-Class Validation mAP50-95")
plt.tight_layout()
plt.show()

# 27. Test set 최종 평가

이제 Validation이 아니라 **Test set**에서 최종 성능을 측정합니다.

핵심은 `split="test"`입니다.

Test 성능을 보고 다시 baseline 설정을 바꾸는 작업을 반복하면
Test set이 사실상 Validation처럼 사용되므로 피해야 합니다.

In [ ]:
test_metrics = best_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    plots=True,
    project=str(EVAL_PROJECT),
    name="test",
    exist_ok=True,
    verbose=True,
)

TEST_SAVE_DIR = Path(test_metrics.save_dir)
print("Test outputs:", TEST_SAVE_DIR)

# 28. Test 핵심 성능지표

최종 baseline 기록에서는 Test의 `mAP50-95`를 가장 중요한 값으로 남기고,
Precision / Recall / mAP50도 함께 기록합니다.

In [ ]:
test_summary = pd.DataFrame([metrics_to_row(test_metrics, "test")])

display(test_summary.style.format({
    "precision": "{:.4f}",
    "recall": "{:.4f}",
    "f1_from_global_P_R": "{:.4f}",
    "mAP50": "{:.4f}",
    "mAP75": "{:.4f}",
    "mAP50_95": "{:.4f}",
    "inference_ms_per_image": "{:.3f}",
}))

# 29. Validation vs Test 비교

Validation보다 Test가 크게 낮다면 다음을 확인할 수 있습니다.

- Validation과 Test 분포 차이
- Validation에 대한 과적합
- Test의 특정 클래스 난이도
- 촬영 환경 / 배경 / 객체 크기 차이

두 성능이 비슷하다면 일반화가 비교적 안정적이라는 근거가 됩니다.

In [ ]:
final_metrics_df = pd.concat([val_summary, test_summary], ignore_index=True)

display(final_metrics_df.style.format({
    "precision": "{:.4f}",
    "recall": "{:.4f}",
    "f1_from_global_P_R": "{:.4f}",
    "mAP50": "{:.4f}",
    "mAP75": "{:.4f}",
    "mAP50_95": "{:.4f}",
    "inference_ms_per_image": "{:.3f}",
}))

In [ ]:
metrics = ["precision", "recall", "mAP50", "mAP50_95"]
labels = ["Precision", "Recall", "mAP50", "mAP50-95"]

val_values = [
    final_metrics_df.loc[final_metrics_df["split"] == "validation", m].iloc[0]
    for m in metrics
]
test_values = [
    final_metrics_df.loc[final_metrics_df["split"] == "test", m].iloc[0]
    for m in metrics
]

x = np.arange(len(metrics))
width = 0.35

plt.figure(figsize=(10, 6))
plt.bar(x - width / 2, val_values, width, label="Validation")
plt.bar(x + width / 2, test_values, width, label="Test")
plt.xticks(x, labels)
plt.ylim(0, 1)
plt.ylabel("Score")
plt.title("YOLO Baseline: Validation vs Test")
plt.legend()
plt.tight_layout()
plt.show()

# 30. Test 결과 시각화

Test에서도 PR Curve와 Confusion Matrix를 확인합니다.

Validation과 비슷한 패턴인지,
혹은 Test에서 새로운 클래스 혼동이 나타나는지 확인하세요.

In [ ]:
test_plot_files = [
    "PR_curve.png",
    "F1_curve.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "val_batch0_pred.jpg",
]

for filename in test_plot_files:
    display_saved_image(TEST_SAVE_DIR, filename)

# 31. Test 클래스별 mAP50-95

전체 Test mAP만 보지 않고,
각 클래스의 성능과 Test 객체 수를 함께 확인합니다.

In [ ]:
test_maps = np.asarray(test_metrics.box.maps, dtype=float)

test_per_class_df = pd.DataFrame({
    "class_id": range(len(test_maps)),
    "class_name": [CLASS_NAMES.get(i, f"class_{i}") for i in range(len(test_maps))],
    "test_mAP50_95": test_maps,
})

test_per_class_df = test_per_class_df.merge(
    class_support_df[
        ["class_id", "train_objects", "val_objects", "test_objects"]
    ],
    on="class_id",
    how="left",
).sort_values("test_mAP50_95", ascending=False)

display(test_per_class_df)

In [ ]:
plot_df = test_per_class_df.sort_values("test_mAP50_95")

plt.figure(figsize=(12, max(8, len(plot_df) * 0.25)))
plt.barh(plot_df["class_name"], plot_df["test_mAP50_95"])
plt.xlim(0, 1)
plt.xlabel("Test mAP50-95")
plt.ylabel("Class")
plt.title("Per-Class Test mAP50-95")
plt.tight_layout()
plt.show()

# 32. Test 이미지 Prediction 직접 확인

정량적인 mAP만으로는 모델의 실제 동작을 모두 알 수 없습니다.

Test 이미지 몇 장을 직접 예측해 다음을 눈으로 확인합니다.

- 실제 객체를 제대로 찾는가?
- bbox 위치가 정확한가?
- 불필요한 영역을 객체로 잘못 검출하지 않는가?
- 비슷한 클래스끼리 혼동하지 않는가?
- 작은 객체 / 여러 객체에서도 잘 동작하는가?

In [ ]:
NUM_PREDICTION_SAMPLES = min(12, len(split_files["test"]))

prediction_results = best_model.predict(
    source=[str(p) for p in split_files["test"][:NUM_PREDICTION_SAMPLES]],
    imgsz=IMGSZ,
    conf=0.25,
    device=DEVICE,
    save=True,
    project=str(PREDICT_PROJECT),
    name="test_samples",
    exist_ok=True,
    verbose=False,
)

print(f"Predicted {len(prediction_results)} test images.")

In [ ]:
for result in prediction_results:
    plotted_bgr = result.plot()
    plotted_rgb = cv2.cvtColor(plotted_bgr, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(10, 7))
    plt.imshow(plotted_rgb)
    plt.title(Path(result.path).name)
    plt.axis("off")
    plt.tight_layout()
    plt.show()

# 33. 최종 결과 CSV 저장

이후 증강 실험이나 다른 모델과 비교할 수 있도록 결과를 파일로 남깁니다.

생성되는 주요 파일:

```text
yolo_baseline_results/
├── baseline_metrics.csv
├── class_support.csv
├── validation_per_class_map.csv
├── test_per_class_map.csv
├── train/
├── evaluation/
└── predictions/
```

In [ ]:
BASELINE_METRICS_CSV = EXPERIMENT_ROOT / "baseline_metrics.csv"
CLASS_SUPPORT_CSV = EXPERIMENT_ROOT / "class_support.csv"
VAL_PER_CLASS_CSV = EXPERIMENT_ROOT / "validation_per_class_map.csv"
TEST_PER_CLASS_CSV = EXPERIMENT_ROOT / "test_per_class_map.csv"

final_metrics_df.to_csv(BASELINE_METRICS_CSV, index=False, encoding="utf-8-sig")
class_support_df.to_csv(CLASS_SUPPORT_CSV, index=False, encoding="utf-8-sig")
val_per_class_df.to_csv(VAL_PER_CLASS_CSV, index=False, encoding="utf-8-sig")
test_per_class_df.to_csv(TEST_PER_CLASS_CSV, index=False, encoding="utf-8-sig")

print("Saved:")
print("-", BASELINE_METRICS_CSV)
print("-", CLASS_SUPPORT_CSV)
print("-", VAL_PER_CLASS_CSV)
print("-", TEST_PER_CLASS_CSV)

# 34. 최종 Baseline 요약 출력

팀 공유나 실험 기록에 바로 옮길 수 있도록
모델 설정과 Validation/Test 핵심 결과를 한 번에 출력합니다.

In [ ]:
val_row = final_metrics_df[final_metrics_df["split"] == "validation"].iloc[0]
test_row = final_metrics_df[final_metrics_df["split"] == "test"].iloc[0]

print("=" * 80)
print("YOLO BASELINE SUMMARY")
print("=" * 80)

print(f"Model            : {MODEL_NAME}")
print(f"Epochs requested : {EPOCHS}")
print(f"Image size       : {IMGSZ}")
print(f"Batch size       : {BATCH}")
print(f"Seed             : {SEED}")
print(f"Device           : {DEVICE}")
print(f"Training time    : {train_minutes:.2f} min")
print(f"Best checkpoint  : {BEST_PT}")

print()
print("[Validation]")
print(f"Precision        : {val_row['precision']:.4f}")
print(f"Recall           : {val_row['recall']:.4f}")
print(f"mAP50            : {val_row['mAP50']:.4f}")
print(f"mAP50-95         : {val_row['mAP50_95']:.4f}")

print()
print("[Test]")
print(f"Precision        : {test_row['precision']:.4f}")
print(f"Recall           : {test_row['recall']:.4f}")
print(f"mAP50            : {test_row['mAP50']:.4f}")
print(f"mAP50-95         : {test_row['mAP50_95']:.4f}")

print("=" * 80)

# 35. 결과 해석 가이드

## 1) mAP50-95
가장 먼저 기록할 대표 Detection 지표입니다.
이후 증강/모델 변경 실험의 1차 비교 기준으로 사용합니다.

## 2) Recall
우리 서비스처럼 이미지 안의 폐기물을 찾아 사용자에게 후보 bbox를 보여주는 흐름에서는
실제 객체를 놓치는 False Negative가 많으면 UX에 직접 영향을 줍니다.
따라서 mAP와 함께 Recall도 중요합니다.

## 3) Precision
Precision이 낮으면 실제 객체가 아닌 영역까지 많이 검출하고 있을 수 있습니다.
Prediction 이미지와 Confusion Matrix를 함께 확인합니다.

## 4) 전체 평균만 보지 않기
클래스가 많으면 전체 평균 뒤에 특정 클래스의 낮은 성능이 숨을 수 있습니다.

함께 봐야 할 것:

- 전체 mAP50-95
- 클래스별 mAP50-95
- 클래스별 Validation/Test 객체 수
- Confusion Matrix
- 실제 Prediction 결과

## 5) Validation vs Test
Validation은 좋은데 Test가 크게 낮으면
데이터 분포 차이 또는 Validation에 대한 과적합 가능성을 점검합니다.

# 36. Baseline 이후 실험 원칙

Baseline 결과가 확정되면 다음 단계에서는 **한 번에 한 요소씩 변경**해 비교하는 것이 좋습니다.

예:

```text
Baseline
  ↓
HSV augmentation
  ↓
Geometry augmentation
  ↓
Mosaic
  ↓
OpenCV augmentation
  ↓
Best augmentation combination
```

각 실험에서 최소한 다음 지표는 같은 표에 기록하세요.

- Precision
- Recall
- mAP50
- mAP50-95

가능하면 같은 seed, 같은 split, 같은 image size, 같은 epoch 조건을 유지해야
변경한 요소 자체의 효과를 비교하기 쉽습니다.

# 37. 실행 전 체크리스트

- [ ] `PROCESSED_DIR`가 실제 processed 폴더를 가리키는가?
- [ ] `data.yaml`이 존재하는가?
- [ ] `data.yaml`에 `train`, `val`, `test`, `names`가 모두 있는가?
- [ ] Train / Validation / Test 이미지와 라벨이 모두 존재하는가?
- [ ] YOLO bbox 좌표가 0~1 정규화 형식인가?
- [ ] Train / Validation / Test에 동일 이미지가 겹치지 않는가?
- [ ] GPU가 의도한 device로 잡혔는가?
- [ ] Baseline 설정을 증강 실험과 섞지 않았는가?
- [ ] Test 결과를 보고 baseline 하이퍼파라미터를 반복 튜닝하지 않는가?

위 항목을 확인했다면 위에서부터 순서대로 실행하면 됩니다.

---

## 참고

이 노트북의 학습/평가 흐름은 Ultralytics의 현재 공식 Python API 구조를 기준으로 작성했습니다.

- pretrained YOLO26 nano 모델 로드
- `model.train(...)`으로 학습
- `model.val(..., split="val")`로 Validation
- `model.val(..., split="test")`로 독립 Test 평가
- Detection 핵심 지표로 Precision, Recall, mAP50, mAP50-95 확인

공식 문서:

- https://docs.ultralytics.com/modes/train/
- https://docs.ultralytics.com/modes/val/
- https://docs.ultralytics.com/tasks/detect/